# 02: Etap 1, MotionNet (globalny ruch kamery)

**Status: wytrenowane 2026-10-13** (stary notatnik `archiwum/train_colab_2026-10-13.ipynb`, Colab T4, 30 epok).
Wagi: `MyDrive/rs_project/runs/2026-10-13_motion_dual/best.pt` (kopia z dawnego `MyDrive/rs_runs/`).

| run | PSNR | SSIM | EPE |
|---|---|---|---|
| bez korekcji | 19.66 | 0.527 | – |
| motion_dual | **28.51** | **0.874** | **0.57 px** |
| motion_single | 19.90 | 0.544 | 3.98 px |
| flow_dual | 20.45 | 0.588 | 3.15 px |

Ten notatnik służy do ponownej ewaluacji i ewentualnego treningu od nowa (np. na zbiorze z parametrami zmierzonymi na kamerach).

In [1]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
kod: f2abceb  all datasets are shown now  (2026-10-24)

NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
P.migrate_old_layout()     # jednorazowo kopiuje stare MyDrive/rs_runs/* -> rs_project/runs/2026-10-13_*
P.status()

dataset zip : /content/drive/MyDrive/rs_dataset_256.zip (1.06 GB)
runs (/content/drive/MyDrive/rs_project/runs):
  2026-10-13_flow_dual             best.pt: tak | results.json: -
  2026-10-13_motion_dual           best.pt: tak | results.json: -
  2026-10-13_motion_single         best.pt: tak | results.json: -


In [3]:
# Zbiór danych: rozpakowanie z Drive na dysk maszyny Colaba (szybki odczyt w trakcie treningu)
DATASET_NAME = 'rs_dataset_256_full'   # 'rs_dataset_256' = stary, mały zbiór z 13.10 (~2,1 tys. par)
if not os.path.exists('/content/dataset/train'):
    z = P.find_dataset_zip(DATASET_NAME)
    assert z, f'Brak zbioru {DATASET_NAME} na Drive - najpierw uruchom 01_dataset.ipynb'
    print('rozpakowuję', z, '...')
    !unzip -q -n "{z}" -d /content
print(len(os.listdir('/content/dataset/train')), 'próbek train /', len(os.listdir('/content/dataset/val')), 'val')
WORKERS = min(8, os.cpu_count())

14540 próbek train / 1460 val


## Trening (uruchamiaj tylko, gdy trenujesz od nowa)
Nazwa runu zawiera datę, więc nowy trening nie nadpisze starego. Po rozłączeniu: dodaj `--resume <folder>/last.pt`.

In [4]:
from datetime import date
RUN = f'{date.today()}_motion_dual'
!python src/train.py --data /content/dataset --mode motion --epochs 30 --bs 16 \
    --workers {WORKERS} --out "{P.RUNS}" --name {RUN}

device: cuda
2026-10-24_motion_dual: 3.58 M params (3.58 M trainable), 14540 train / 1460 val samples
/content/code/src/train.py:304: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()
ep   1/30 loss 7.1877 | val PSNR 23.75 dB (input 20.23) SSIM 0.7123 (input 0.5475) EPE 1.84px | B-only border PSNR 16.58 (input 14.85) | 107s
ep   2/30 loss 2.3722 | val PSNR 25.80 dB (input 20.23) SSIM 0.7867 (input 0.5475) EPE 1.21px | B-only border PSNR 17.19 (input 14.85) | 103s
ep   3/30 loss 1.6073 | val PSNR 27.64 dB (input 20.23) SSIM 0.8460 (input 0.5475) EPE 0.83px | B-only border PSNR 17.32 (input 14.85) | 104s
ep   4/30 loss 1.2259 | val PSNR 27

In [5]:
# ablacje (do pracy): tylko kamera A, oraz U-Net bez geometrii
!python src/train.py --data /content/dataset --mode motion --single --epochs 30 --bs 16 --workers {WORKERS} --out "{P.RUNS}" --name {date.today()}_motion_single
!python src/train.py --data /content/dataset --mode flow --epochs 30 --bs 16 --workers {WORKERS} --out "{P.RUNS}" --name {date.today()}_flow_dual

device: cuda
2026-10-24_motion_single: 3.48 M params (3.48 M trainable), 14540 train / 1460 val samples
/content/code/src/train.py:304: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()
ep   1/30 loss 14.7674 | val PSNR 20.39 dB (input 20.23) SSIM 0.5590 (input 0.5475) EPE 4.00px | B-only border PSNR 14.93 (input 14.85) | 84s
ep   2/30 loss 14.7751 | val PSNR 20.46 dB (input 20.23) SSIM 0.5655 (input 0.5475) EPE 4.01px | B-only border PSNR 14.90 (input 14.85) | 83s
ep   3/30 loss 14.7594 | val PSNR 20.39 dB (input 20.23) SSIM 0.5593 (input 0.5475) EPE 4.00px | B-only border PSNR 14.90 (input 14.85) | 82s
ep   4/30 loss 14.7356 | val PSNR

## Ewaluacja wszystkich runów etapu 1 (zapisuje `eval.json` w folderze runu)

In [6]:
import json, pandas as pd
rows = []
for name in ['motion_dual', 'motion_single', 'flow_dual']:
    ck = P.find_run(name)
    if ck is None:
        print('brak', name); continue
    out = ck.parent / 'eval.json'
    !python src/train.py --data /content/dataset --eval "{ck}" --eval-json "{out}" --workers {WORKERS} > /dev/null
    m = json.load(open(out))['metrics']
    rows.append({'run': ck.parent.name, **{k: round(v, 3) for k, v in m.items()}})
pd.DataFrame(rows).set_index('run')

,psnr,ssim,psnr_in,ssim_in,epe,epe_in,psnr_b,psnr_b_in,ms_per_frame,params_M
run,,,,,,,,,,
2026-10-24_motion_dual,31.199,0.914,20.232,0.547,0.333,4.004,17.727,14.847,12.976,3.579
2026-10-24_motion_single,20.460,0.566,20.232,0.547,4.005,4.004,14.905,14.847,6.840,3.481
2026-10-24_flow_dual,28.398,0.870,20.232,0.547,0.703,4.004,17.275,14.847,2.080,2.723
